# Legacy AIM-UNet: train on 10 real LiTS volumes (5-fold CV), test on 4 synthetic volumes

**Runtime -> Change runtime type -> GPU (T4 or better)** before running.

1. Upload to Google Drive:
   * the 10 LiTS volumes `volume-<i>.nii` + `segmentation-<i>.nii` listed in `TRAIN_LITS` (LiTS Training Batch 1, ids 0-22)
   * the 4 synthetic volumes in `TEST_SYN` (`synthetic_XX_<host>_r<R>mm.nii.gz` + `..._label.nii.gz`, from `results/synthetic_healthy40/images`)
   Each set can sit anywhere under its folder (any subfolders): the notebook searches the folder for the files by name.
2. Set `LITS_ROOT` (original LiTS), `SYN_ROOT` (synthetic) and `OUT_DIR` in the next cell.
3. Run all cells (Runtime -> Run all).

Model, loss (CE weights [1,2,1] + soft Dice), Adam lr 1e-4, batch 4, flip/rotate augmentation and tumour-slice
oversampling are the legacy `train_orig_AIMUNetALF.py`. Slices are axial with a fixed HU window -100..250.
Synthetic labels: 0 background, 1 liver (host's liver mask), 2 tumour (inserted GAN tumour), same as LiTS.

Split by volume: the 10 LiTS volumes go into 5-fold CV (8 train / 2 validation). Each fold trains 4 epochs and saves the
model after every epoch; its final model is scored on the 4 synthetic test volumes (pooled and per volume).
Outputs: `train_AIMUNet.log`, `history.csv`, `test_per_fold.csv`, `test_per_volume.csv`, `summary.json`, ROC curves, `curves.png`.

In [ ]:
# ===== 1. Connect Google Drive and set paths (EDIT THESE) =====
from google.colab import drive
drive.mount('/content/drive')

LITS_ROOT = '/content/drive/MyDrive/<PATH_TO_ORIGINAL_LITS_FOLDER>'         # TRAIN/VAL: searched (with subfolders) for volume-<i> + segmentation-<i>
SYN_ROOT  = '/content/drive/MyDrive/<PATH_TO_SYNTHETIC_FOLDER>'             # TEST: searched (with subfolders) for synthetic_XX_... + _label
OUT_DIR   = '/content/drive/MyDrive/<PATH_TO>/aimunet_lits10_syn4'          # results + checkpoints are written here

In [ ]:
!pip -q install nibabel scikit-learn
!nvidia-smi -L

In [ ]:
# ===== 2. Settings =====
EPOCHS     = 4       # per fold
FOLDS      = 5       # cross-validation folds over the LiTS volumes
SIZE      = 256     # slice size (legacy = 256)
BATCH      = 4       # legacy = 4
LR         = 1e-4
STEPS      = None    # batches per epoch; None = one pass over the training slices (legacy)
MARGIN     = 3       # liver slices +- MARGIN are used; None = every slice of the volume (legacy, slower)
WIN_LO, WIN_HI = -100.0, 250.0
AUC_PIXELS = 400_000 # pixels sampled for AUC / ROC (all tumour pixels are always kept)
SEED       = 42
# 10 LiTS Training Batch 1 volumes (ids 0-22), all with tumour, mixed tumour size and slice thickness; none is a test host
TRAIN_LITS = [0, 2, 6, 8, 10, 13, 14, 16, 19, 21]
# 4 synthetic test volumes: 3 CHAOS hosts + 1 LiTS host, tumour radius 10 / 15 / 5 / 15 mm
TEST_SYN = ["synthetic_02_chaos-2_r10mm", "synthetic_03_chaos-8_r15mm", "synthetic_10_chaos-24_r5mm",
            "synthetic_18_lits-91_r15mm"]

In [ ]:
# ===== 3. Code: data, legacy AIM-UNet, loss, metrics =====
import os, glob, time, json, csv, random, logging
import numpy as np, nibabel as nib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt

os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ('background', 'liver', 'tumor')
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(os.path.join(OUT_DIR, 'train_AIMUNet.log'), mode='w'), logging.StreamHandler()])
log = logging.getLogger('aimunet_cv')

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def wanted(v):
    """(image stem, mask stem) of one case: int = LiTS case id, str = synthetic name."""
    return (f'volume-{v}', f'segmentation-{v}') if isinstance(v, int) else (v, v + '_label')

def find_files(root, cases):
    """Search root and all its subfolders once; -> {case: (image path, mask path)}. Accepts .nii and .nii.gz.
    A name found in several folders: the first one (sorted path) is used and the others are reported."""
    stems = {s for v in cases for s in wanted(v)}
    hits = {}
    for d, sub, files in os.walk(root):
        sub.sort()
        for f in sorted(files):
            stem = f[:-7] if f.endswith('.nii.gz') else f[:-4] if f.endswith('.nii') else None
            if stem in stems:
                hits.setdefault(stem, []).append(os.path.join(d, f))
    missing = sorted(s for s in stems if s not in hits)
    if missing:
        raise FileNotFoundError(f'not found anywhere under {root}: ' + ', '.join(m + '.nii[.gz]' for m in missing))
    for s, p in hits.items():
        if len(p) > 1:
            print(f'{s}: found {len(p)} copies, using {p[0]}')
    return {v: tuple(hits[s][0] for s in wanted(v)) for v in cases}

def name(v):
    return f'lits-{v}' if isinstance(v, int) else v

FILES = {**find_files(LITS_ROOT, TRAIN_LITS), **find_files(SYN_ROOT, TEST_SYN)}
for v, (ip, mp) in FILES.items():
    print(f'{name(v):28s} {ip}\n{"":28s} {mp}')
paths = FILES.__getitem__

def load_volume(v):
    """-> list of (image (SIZE,SIZE,1) float [0,1], mask (SIZE,SIZE,1) uint8), axial slices."""
    ip, mp = paths(v)
    vol = np.asarray(nib.load(ip).dataobj, dtype=np.float32)
    seg = np.asarray(nib.load(mp).dataobj).astype(np.uint8)
    if MARGIN is None:
        z = np.arange(seg.shape[2])
    else:
        zl = np.flatnonzero((seg > 0).any((0, 1)))
        z = np.arange(max(zl[0] - MARGIN, 0), min(zl[-1] + MARGIN + 1, seg.shape[2]))
    x = torch.from_numpy(np.clip((vol[:, :, z] - WIN_LO) / (WIN_HI - WIN_LO), 0, 1)).permute(2, 0, 1)[:, None]
    y = torch.from_numpy(seg[:, :, z]).permute(2, 0, 1)[:, None].float()
    x = F.interpolate(x, size=(SIZE, SIZE), mode='bilinear', antialias=True, align_corners=False)
    y = F.interpolate(y, size=(SIZE, SIZE), mode='nearest')
    x, y = x.permute(0, 2, 3, 1).numpy(), y.permute(0, 2, 3, 1).numpy().astype(np.uint8)
    return list(zip(x, y))

class LiverDataset(Dataset):  # legacy __getitem__ (PIL flips + rotation in [-10, 10] deg)
    def __init__(self, data, augment=False):
        self.data, self.augment = data, augment
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        img, msk = self.data[idx]
        img_pil = Image.fromarray((img[..., 0] * 255).astype(np.uint8))
        msk_pil = Image.fromarray(msk[..., 0].astype(np.uint8))
        if self.augment:
            if random.random() > 0.5:
                img_pil, msk_pil = TF.hflip(img_pil), TF.hflip(msk_pil)
            if random.random() > 0.5:
                img_pil, msk_pil = TF.vflip(img_pil), TF.vflip(msk_pil)
            angle = random.uniform(-10, 10)
            img_pil = TF.rotate(img_pil, angle, interpolation=InterpolationMode.BILINEAR)
            msk_pil = TF.rotate(msk_pil, angle, interpolation=InterpolationMode.NEAREST)
        return TF.to_tensor(img_pil), torch.as_tensor(np.array(msk_pil), dtype=torch.long)

# ---- legacy AIM-UNet ----
class InceptionModule(nn.Module):
    def __init__(self, in_c):
        super().__init__()
        self.branch1 = nn.Sequential(nn.Conv2d(in_c, in_c, 1), nn.BatchNorm2d(in_c), nn.ReLU())
        self.branch2 = nn.Sequential(nn.Conv2d(in_c, in_c, 1), nn.Conv2d(in_c, in_c, 5, padding=2), nn.BatchNorm2d(in_c), nn.ReLU())
        self.branch3 = nn.Sequential(nn.Conv2d(in_c, in_c, 1), nn.Conv2d(in_c, in_c, 3, padding=1),
                                     nn.Conv2d(in_c, in_c, 3, padding=1), nn.BatchNorm2d(in_c), nn.ReLU())
        self.branch4 = nn.Sequential(nn.MaxPool2d(3, stride=1, padding=1), nn.Conv2d(in_c, in_c, 1), nn.BatchNorm2d(in_c), nn.ReLU())
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(), nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
    def forward(self, x):
        return self.conv(x)

class AIM_UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1, self.pool1 = ConvBlock(1, 64), nn.MaxPool2d(2)
        self.enc2, self.pool2 = ConvBlock(64, 128), nn.MaxPool2d(2)
        self.enc3, self.pool3 = ConvBlock(128, 256), nn.MaxPool2d(2)
        self.enc4, self.pool4 = ConvBlock(256, 512), nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(512, 1024)
        self.inc4, self.up4, self.dec4 = InceptionModule(512), nn.ConvTranspose2d(1024, 512, 2, 2), ConvBlock(512 * 5, 512)
        self.inc3, self.up3, self.dec3 = InceptionModule(256), nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(256 * 5, 256)
        self.inc2, self.up2, self.dec2 = InceptionModule(128), nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(128 * 5, 128)
        self.inc1, self.up1, self.dec1 = InceptionModule(64), nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(64 * 5, 64)
        self.out = nn.Conv2d(64, 3, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool1(e1)); e3 = self.enc3(self.pool2(e2)); e4 = self.enc4(self.pool3(e3))
        b = self.bottleneck(self.pool4(e4))
        d4 = self.dec4(torch.cat([self.inc4(e4), self.up4(b)], 1))
        d3 = self.dec3(torch.cat([self.inc3(e3), self.up3(d4)], 1))
        d2 = self.dec2(torch.cat([self.inc2(e2), self.up2(d3)], 1))
        d1 = self.dec1(torch.cat([self.inc1(e1), self.up1(d2)], 1))
        return self.out(d1)

# ---- legacy loss: CE (weights 1,2,1) + soft Dice ----
ce_loss = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0, 1.0], device=device))
def dice_loss(logits, target, smooth=1e-6):
    probs = F.softmax(logits, dim=1)
    one_hot = F.one_hot(target, num_classes=3).permute(0, 3, 1, 2).float()
    inter, union = (probs * one_hot).sum((0, 2, 3)), probs.sum((0, 2, 3)) + one_hot.sum((0, 2, 3))
    return 1.0 - ((2 * inter + smooth) / (union + smooth)).mean()
loss_fn = lambda out, y: ce_loss(out, y) + dice_loss(out, y)

def confusion_gpu(pred, target):
    return torch.bincount((target.reshape(-1) * 3 + pred.reshape(-1)), minlength=9).reshape(3, 3).cpu().numpy()

def dice_from(cm):
    tp = np.diag(cm)
    return {f'dice_{c}': float(2 * tp[i] / max(cm[i].sum() + cm[:, i].sum(), 1)) for i, c in enumerate(CLASSES)}

def metrics_from(cm, scores, truth):
    tp = np.diag(cm).astype(float)
    m = dice_from(cm)
    m['accuracy'] = float(tp.sum() / cm.sum())
    m['balanced_accuracy'] = float(np.mean(tp / np.maximum(cm.sum(1), 1)))
    for i, c in enumerate(CLASSES):
        fn, fp = cm[i].sum() - tp[i], cm[:, i].sum() - tp[i]
        m[f'sens_{c}'] = float(tp[i] / max(tp[i] + fn, 1))
        m[f'spec_{c}'] = float((cm.sum() - tp[i] - fn - fp) / max(cm.sum() - tp[i] - fn, 1))
        pos = truth == i
        m[f'auc_{c}'] = float(roc_auc_score(pos, scores[:, i])) if 0 < pos.sum() < len(pos) else float('nan')
    m['auc_macro'] = float(np.nanmean([m[f'auc_{c}'] for c in CLASSES]))
    return m

@torch.no_grad()
def evaluate(model, vols, data):
    """Loss + metrics pooled over every slice of vols, plus per-volume Dice.
    AUC/ROC on all tumour pixels + a random sample of the others."""
    model.eval()
    frac = min(1.0, AUC_PIXELS / max(sum(len(data[v]) for v in vols) * SIZE * SIZE, 1))
    cm, run, n, S, T, per = np.zeros((3, 3), np.int64), 0.0, 0, [], [], {}
    for v in vols:
        cv = np.zeros((3, 3), np.int64)
        for xb, yb in DataLoader(LiverDataset(data[v]), batch_size=8, shuffle=False, num_workers=2):
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            run += loss_fn(out, yb).item() * len(xb); n += len(xb)
            cv += confusion_gpu(out.argmax(1), yb)
            p = F.softmax(out, 1).permute(0, 2, 3, 1).reshape(-1, 3)
            t = yb.reshape(-1)
            keep = (t == 2) | (torch.rand(t.shape, device=device) < frac)
            S.append(p[keep].cpu().numpy()); T.append(t[keep].cpu().numpy())
        cm += cv
        per[name(v)] = dice_from(cv)
    m = metrics_from(cm, np.concatenate(S), np.concatenate(T))
    m['loss'] = run / n
    return m, cm, (np.concatenate(S), np.concatenate(T)), per

def plot_roc(path, sc, title):
    scores, truth = sc
    plt.figure(figsize=(5, 5))
    for i, c in enumerate(CLASSES):
        pos = truth == i
        if 0 < pos.sum() < len(pos):
            fpr, tpr, _ = roc_curve(pos, scores[:, i])
            plt.plot(fpr, tpr, label=f'{c} (AUC {roc_auc_score(pos, scores[:, i]):.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
    plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.title(title); plt.legend(loc='lower right')
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.close()

def fmt(m):
    return (f"loss {m['loss']:.4f} | Dice bg {m['dice_background']:.4f} liver {m['dice_liver']:.4f} tumor {m['dice_tumor']:.4f} | "
            f"acc {m['accuracy']:.4f} bal-acc {m['balanced_accuracy']:.4f} | AUC bg {m['auc_background']:.4f} "
            f"liver {m['auc_liver']:.4f} tumor {m['auc_tumor']:.4f} macro {m['auc_macro']:.4f}")

In [ ]:
# ===== 4. Training: one fold =====
def train_fold(k, tr, va, data):
    fd = os.path.join(OUT_DIR, f'fold{k}')
    os.makedirs(fd, exist_ok=True)
    seed_all(SEED + k)
    train_ds = LiverDataset([s for v in tr for s in data[v]], augment=True)
    has_t = np.array([(m[..., 0] == 2).any() for _, m in train_ds.data])
    w = np.where(has_t, (~has_t).sum() / max(has_t.sum(), 1), 1.0)   # legacy tumour-slice oversampling
    n_samples = STEPS * BATCH if STEPS else len(w)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=WeightedRandomSampler(w.tolist(), n_samples, replacement=True),
                          num_workers=2, pin_memory=device.type == 'cuda')
    log.info(f'FOLD {k}: train {[name(v) for v in tr]} ({len(train_ds)} slices, {has_t.sum()} with tumour) | '
             f'val {[name(v) for v in va]} ({sum(len(data[v]) for v in va)} slices)')
    model = AIM_UNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    rng, hist = np.random.default_rng(SEED + k), []
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        run, cm = 0.0, np.zeros((3, 3), np.int64)
        for xb, yb in train_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            run += loss.item()
            cm += confusion_gpu(out.detach().argmax(1), yb)
        tp = np.diag(cm)
        trm = dict(loss=run / len(train_dl), accuracy=float(tp.sum() / cm.sum()),
                   **{f'dice_{c}': float(2 * tp[i] / max(cm[i].sum() + cm[:, i].sum(), 1)) for i, c in enumerate(CLASSES)})
        vm, vcm, vsc, _ = evaluate(model, va, data)
        ck = os.path.join(fd, f'aimunet_fold{k}_epoch{ep:02d}.pt')
        torch.save({'model': model.state_dict(), 'epoch': ep, 'fold': k, 'train': [name(v) for v in tr],
                    'val': [name(v) for v in va], 'val_metrics': vm, 'size': SIZE, 'window': [WIN_LO, WIN_HI]}, ck)
        plot_roc(os.path.join(fd, f'roc_val_epoch{ep:02d}.png'), vsc, f'Fold {k} validation (LiTS), epoch {ep}')
        log.info(f"[fold {k}] Epoch {ep}/{EPOCHS} TRAIN loss {trm['loss']:.4f} | Dice bg {trm['dice_background']:.4f} "
                 f"liver {trm['dice_liver']:.4f} tumor {trm['dice_tumor']:.4f} | acc {trm['accuracy']:.4f}")
        log.info(f'[fold {k}] Epoch {ep}/{EPOCHS} VAL   {fmt(vm)}')
        log.info(f'[fold {k}] Epoch {ep}/{EPOCHS} VAL confusion (rows true bg/liver/tumor):\n{vcm}')
        log.info(f'[fold {k}] Epoch {ep}/{EPOCHS} saved {ck} ({time.time() - t0:.0f}s)')
        hist.append({'fold': k, 'epoch': ep, **{f'train_{a}': b for a, b in trm.items()}, **{f'val_{a}': b for a, b in vm.items()}})
    tm, tcm, tsc, per = evaluate(model, TEST_SYN, data)
    plot_roc(os.path.join(fd, 'roc_test_final.png'), tsc, f'Fold {k} model on synthetic test set')
    log.info(f'[fold {k}] TEST synthetic (epoch {EPOCHS} model) {fmt(tm)}')
    log.info(f'[fold {k}] TEST confusion:\n{tcm}')
    for v, d in per.items():
        log.info(f"[fold {k}] TEST {v}: Dice liver {d['dice_liver']:.4f} tumor {d['dice_tumor']:.4f}")
    return hist, tm, tsc, per

In [ ]:
# ===== 5. Run: load data, split, cross-validate, test =====
t_all = time.time()
seed_all(SEED)
log.info(f'AIM-UNet (legacy): train/val on {len(TRAIN_LITS)} LiTS volumes ({FOLDS}-fold CV), test on {len(TEST_SYN)} '
         f'synthetic | {EPOCHS} epochs | {SIZE}px | HU [{WIN_LO}, {WIN_HI}] | {device}')
data = {}
for v in TRAIN_LITS + TEST_SYN:
    data[v] = load_volume(v)
    log.info(f'loaded {name(v)}: {len(data[v])} slices, tumour slices {sum(int((m == 2).any()) for _, m in data[v])}')
log.info(f'TEST (synthetic, held out from every fold): {TEST_SYN}')
split, hist, tests, tsc_all, per_rows = {'test': TEST_SYN, 'folds': []}, [], [], [], []
for k, (i_tr, i_va) in enumerate(KFold(FOLDS, shuffle=True, random_state=SEED).split(TRAIN_LITS), 1):
    tr, va = [TRAIN_LITS[i] for i in i_tr], [TRAIN_LITS[i] for i in i_va]
    split['folds'].append({'fold': k, 'train': [name(v) for v in tr], 'val': [name(v) for v in va]})
    h, tm, tsc, per = train_fold(k, tr, va, data)
    hist += h; tests.append({'fold': k, **tm}); tsc_all.append(tsc)
    per_rows += [{'fold': k, 'volume': v, **d} for v, d in per.items()]
    json.dump({'split': split, 'history': hist, 'test': tests, 'test_per_volume': per_rows},
              open(os.path.join(OUT_DIR, 'history.json'), 'w'), indent=1)

In [ ]:
# ===== 6. Summary tables, curves, pooled test ROC =====
for fn, rows in (('history.csv', hist), ('test_per_fold.csv', tests), ('test_per_volume.csv', per_rows)):
    with open(os.path.join(OUT_DIR, fn), 'w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
keys = ['loss', 'dice_background', 'dice_liver', 'dice_tumor', 'accuracy', 'balanced_accuracy',
        'auc_background', 'auc_liver', 'auc_tumor', 'auc_macro']
ms = lambda xs: [float(np.nanmean(xs)), float(np.nanstd(xs))]
last = [h for h in hist if h['epoch'] == EPOCHS]
summary = {'split': split, 'epochs': EPOCHS, 'size': SIZE,
           'cv_val_lits_final_epoch': {k: ms([h[f'val_{k}'] for h in last]) for k in keys},
           'test_synthetic': {k: ms([t[k] for t in tests]) for k in keys},
           'test_synthetic_per_volume': {v: {c: ms([r[f'dice_{c}'] for r in per_rows if r['volume'] == v]) for c in ('liver', 'tumor')}
                                         for v in TEST_SYN},
           'minutes': round((time.time() - t_all) / 60, 1)}
json.dump(summary, open(os.path.join(OUT_DIR, 'summary.json'), 'w'), indent=1)
log.info(f'SUMMARY (mean +- std over {FOLDS} folds, epoch {EPOCHS}): validation = real LiTS, test = synthetic')
for k in keys:
    a, b = summary['cv_val_lits_final_epoch'][k], summary['test_synthetic'][k]
    log.info(f'  {k:18s} val(LiTS) {a[0]:.4f} +- {a[1]:.4f} | test(synthetic) {b[0]:.4f} +- {b[1]:.4f}')
for v, d in summary['test_synthetic_per_volume'].items():
    log.info(f"  {v:28s} Dice liver {d['liver'][0]:.4f} +- {d['liver'][1]:.4f} | tumor {d['tumor'][0]:.4f} +- {d['tumor'][1]:.4f}")

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for k in range(1, FOLDS + 1):
    h = [r for r in hist if r['fold'] == k]; e = [r['epoch'] for r in h]
    ax[0].plot(e, [r['train_loss'] for r in h], '--', c=f'C{k}'); ax[0].plot(e, [r['val_loss'] for r in h], c=f'C{k}', label=f'fold {k}')
    for j, c in enumerate(('liver', 'tumor')):
        ax[1 + j].plot(e, [r[f'val_dice_{c}'] for r in h], c=f'C{k}', label=f'fold {k}')
    ax[3].plot(e, [r['val_auc_macro'] for r in h], c=f'C{k}', label=f'fold {k}')
for a, t in zip(ax, ('Loss (dashed train, solid val)', 'Val Dice liver', 'Val Dice tumor', 'Val macro AUC')):
    a.set_title(t); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'curves.png'), dpi=110); plt.show()
plot_roc(os.path.join(OUT_DIR, 'roc_test_all_folds.png'),
         (np.concatenate([s for s, _ in tsc_all]), np.concatenate([t for _, t in tsc_all])), 'Synthetic test set, all fold models pooled')
from IPython.display import Image as Show, display
display(Show(os.path.join(OUT_DIR, 'roc_test_all_folds.png')))
log.info(f"ALL DONE in {summary['minutes']} min -> {OUT_DIR}")

In [ ]:
# ===== 7. (optional) Each synthetic test volume: slice with the most tumour, ground truth vs fold-1 final model =====
m = AIM_UNet().to(device)
m.load_state_dict(torch.load(os.path.join(OUT_DIR, 'fold1', f'aimunet_fold1_epoch{EPOCHS:02d}.pt'), map_location=device)['model'])
m.eval()
fig, ax = plt.subplots(len(TEST_SYN), 3, figsize=(12, 4 * len(TEST_SYN)), squeeze=False)
for r, v in enumerate(TEST_SYN):
    ds = LiverDataset(data[v])
    idx = max(range(len(ds)), key=lambda i: int((ds.data[i][1] == 2).sum()))
    x, y = ds[idx]
    with torch.no_grad():
        p = m(x[None].to(device)).argmax(1)[0].cpu().numpy()
    for a, im, t in zip(ax[r], (x[0].numpy(), y.numpy(), p), ('CT (HU -100..250)', 'ground truth', 'prediction')):
        a.imshow(im, cmap='gray' if t.startswith('CT') else 'viridis', vmin=0, vmax=1 if t.startswith('CT') else 2)
        a.set_title(f'{v}\n{t}' if t.startswith('CT') else t); a.axis('off')
plt.tight_layout(); plt.show()